In [0]:
users = spark.table("workspace.default.silver_users")
devices = spark.table("workspace.default.silver_devices")
login = spark.table("workspace.default.silver_login_logs")
threats = spark.table("workspace.default.silver_threat_alerts")

print(users.columns)
print(devices.columns)
print(login.columns)
print(threats.columns)

['UserID', 'UserName', 'Department', 'Country', 'AccountStatus']
['DeviceID', 'UserID', 'DeviceType', 'OperatingSystem', 'TrustDevice']
['LoginID', 'UserID', 'DeviceID', 'LoginTime', 'IPAddress', 'Country', 'LoginMethod', 'LoginStatus', 'RiskLevel']
['AlertID', 'LoginID', 'UserID', 'AlertType', 'AlertSeverity']


In [0]:
# Read Silver Tables
users = spark.table("workspace.default.silver_users")
devices = spark.table("workspace.default.silver_devices")
login = spark.table("workspace.default.silver_login_logs")
threats = spark.table("workspace.default.silver_threat_alerts")

# Remove duplicate columns before join
users = users.drop("Country")
devices = devices.drop("UserID")
threats = threats.drop("UserID")

# Create Gold Table
gold_df = login.join(users, on="UserID", how="left")

gold_df = gold_df.join(devices, on="DeviceID", how="left")

gold_df = gold_df.join(threats, on="LoginID", how="left")

# Check Data
gold_df.show()

# Save Gold Table
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_cybersecurity_dashboard")

print("✅ Gold Layer Created Successfully!")

+-------+--------+------+-------------------+---------------+-------+-----------+-----------+---------+---------------+-----------+-------------+----------+--------------------+-----------+-------+----------------+-------------+
|LoginID|DeviceID|UserID|          LoginTime|      IPAddress|Country|LoginMethod|LoginStatus|RiskLevel|       UserName| Department|AccountStatus|DeviceType|     OperatingSystem|TrustDevice|AlertID|       AlertType|AlertSeverity|
+-------+--------+------+-------------------+---------------+-------+-----------+-----------+---------+---------------+-----------+-------------+----------+--------------------+-----------+-------+----------------+-------------+
|      3|     119|    39|2026-04-07 12:22:00|   9.222.219.96|  India|        MFA|    Success|      Low|Rahul Chaudhary|   Security|       Active|    Mobile|['Windows', 'macO...|    Trusted|   NULL|            NULL|         NULL|
|     34|     102|    91|2026-01-02 07:50:00| 237.48.233.202|     UK|        MFA|   